In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
N = 100
h = 1
base_d = 0.02
base_w = base_d * 10
base_triArea = 0.001
numSegments = 100
tilt_n = 20
total_width = base_d + base_w
clipping_lN = 50
clipping_uN = 53

In [ ]:
# N = 3
# clipping_lN = None
# clipping_uN = None

In [ ]:
freq = 2
amplitude = 1

In [ ]:
use_non_empty_wall = True

In [ ]:
from ipywidgets import interactive, widgets
def plotForAlpha(freq, amplitude, tilt_n): visualization.plot_line_segments(*parametric_pillows.sinusoid_raw(N = N, h = h, d = base_d / freq if use_non_empty_wall else 0, w = base_w / freq, triArea = base_triArea / freq**2, numSegments = int(numSegments * freq * 0.5), freq = np.pi * freq, amplitude = amplitude, tilt_n = tilt_n, clipping_lN = clipping_lN, clipping_uN = clipping_uN))
iplot = interactive(plotForAlpha, 
                    freq = widgets.FloatSlider(min=1, max=10, value=2, step=1), 
                    amplitude = widgets.FloatSlider(min=0.00, max=2, value=0.1, step=0.01), 
                    tilt_n = widgets.IntSlider(min=1, max=30, value=1, step=1))
iplot.children[-1].layout.height = '500px'
display(iplot)

In [ ]:
freq = iplot.children[0].value
amplitude = iplot.children[1].value
tilt_n = iplot.children[2].value

In [ ]:
# Parameters
N = N
h = h
d = base_d / freq if use_non_empty_wall else 0
w = base_w / freq
triArea = base_triArea / freq**2 * base_w * (1 / amplitude ** 0.5)
numSegments = int(numSegments * freq)
freq = np.pi * freq
amplitude = amplitude
tilt_n = tilt_n

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
pts, edges = parametric_pillows.sinusoid_raw(N = N, h = h, d = d, w = w, triArea = triArea, numSegments = numSegments, freq = freq, amplitude = amplitude, tilt_n = tilt_n, clipping_lN = clipping_lN, clipping_uN = clipping_uN, target_length = None)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize = (10, 10))
plt.scatter(np.array(pts)[:, 0], np.array(pts)[:, 1])
plt.axis("equal")
plt.savefig('pattern.png')

In [ ]:
import sheet_meshing

In [ ]:
importlib.reload(sheet_meshing)

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
m, fuseMarkers, brdyWallMarkers = parametric_pillows.sinusoid(N = N, h = h, d = d, w = w, triArea = triArea * 0.5, numSegments = int(numSegments), freq = freq, amplitude = amplitude, tilt_n = tilt_n, clipping_lN = clipping_lN, clipping_uN = clipping_uN, target_length = None, use_periodic = True, epsilon = 1e-3 / freq)

visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=10, height=10)

In [ ]:
isheet = inflation.InflatableSheet(m, np.array(fuseMarkers) != 0)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) != 0, epsilon = 1e-5)

In [ ]:
fixedVars = get_center_fixedVars(ipu)

In [ ]:
# isheet.setRelaxedStiffnessEpsilon(1e-6)

In [ ]:
import py_newton_optimizer
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
viewer.showWireframe(True)

In [ ]:
ipu.sheet.rigidMotionPinVars

In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 10
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixVars, opts, callback=cb)
benchmark.report()

In [ ]:
import compute_vibrational_modes

In [ ]:
class ModalAnalysisWrapper:
    def __init__(self, sheet):
        self.sheet = sheet
    def hessian(self):
        return self.sheet.hessian(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(ipu), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=16, sigma=-1e-10, fixedVars = fixVars)

import mode_viewer, importlib
importlib.reload(mode_viewer);
mview = mode_viewer.ModeViewer(ipu, modes, lambdas, amplitude=10)
# mview.showScalarField(rod_colors)
mview.show()

In [ ]:
modes.shape

In [ ]:
isheet.numVars()

### Finite difference validation

In [ ]:
fd_perturb = np.random.uniform(-1e-5, 1e-5, ipu.numVars())
fd_perturb[:3] *= 0

In [ ]:
ipu.setVars(ipu.getVars() + fd_perturb)

In [ ]:
viewer.update()

In [ ]:
import fd_validation

In [ ]:
ipu.energy(energyType = Elastic)

In [ ]:
Pressure = inflation.InflatableSheet.EnergyType.Pressure
Elastic  = inflation.InflatableSheet.EnergyType.Elastic
Full     = inflation.InflatableSheet.EnergyType.Full

In [ ]:
fd_validation.gradConvergencePlot(ipu, customArgs = {"etype": Elastic}, epsilons = )

In [ ]:
fd_validation.hessConvergencePlot(ipu, customArgs = {"etype": Elastic})

In [ ]:
isheet.numVars()

In [ ]:
vertices = np.reshape(isheet.getVars(), (int(isheet.numVars() / 3), 3))

In [ ]:
# rotation around x axis
x_rotation_vector = []
for vx in vertices:
    x_rotation_vector.append([0, -vx[2], vx[1]])
x_rotation_vector = np.array(x_rotation_vector).flatten()

# rotation around y axis
y_rotation_vector = []
for vx in vertices:
    y_rotation_vector.append([vx[2], 0, -vx[0]])
y_rotation_vector = np.array(y_rotation_vector).flatten()

# rotation around z axis
z_rotation_vector = []
for vx in vertices:
    z_rotation_vector.append([-vx[1], vx[0], 0])
z_rotation_vector = np.array(z_rotation_vector).flatten()

In [ ]:
import numpy.linalg as la

In [ ]:
ortho_x = x_rotation_vector / la.norm(x_rotation_vector)
ortho_y = y_rotation_vector / la.norm(y_rotation_vector)
ortho_z = z_rotation_vector / la.norm(z_rotation_vector)

In [ ]:
for i in range(3):
    ortho_x = (ortho_x - (ortho_x @ ortho_y) / la.norm(ortho_y)**2 * ortho_y)
    ortho_z = (ortho_z - (ortho_z @ ortho_y) / la.norm(ortho_y)**2 * ortho_y)
    ortho_z = (ortho_z - (ortho_z @ ortho_x) / la.norm(ortho_x)**2 * ortho_x)
    ortho_x /= la.norm(ortho_x)
    ortho_y /= la.norm(ortho_y)
    ortho_z /= la.norm(ortho_z)

In [ ]:
ortho_y @ ortho_z, ortho_y @ ortho_x, ortho_x @ ortho_z

In [ ]:
import sparse_matrices

In [ ]:
H = isheet.hessian(energyType = inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
def H_quadratic(x):
    return x @ H.apply(x) / (x @ x)

In [ ]:
H_quadratic(ortho_x), H_quadratic(ortho_y), H_quadratic(ortho_z)

In [ ]:
g = isheet.gradient(energyType = inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
g @ ortho_x, g @ ortho_y, g @ ortho_z

In [ ]:
translation_vector = np.ones(isheet.numVars())

In [ ]:
translation_vector @ H.apply(translation_vector) / (translation_vector @ translation_vector)

In [ ]:
ortho_x

In [ ]:
Hx = H.apply(ortho_x)

In [ ]:
Hx / la.norm(Hx)

In [ ]:
modes = np.array([ortho_x, ortho_y, ortho_z])

In [ ]:
modes = modes.transpose()

In [ ]:
lambdas = [H_quadratic(ortho_x), H_quadratic(ortho_y), H_quadratic(ortho_z)]

In [ ]:
import mode_viewer, importlib
importlib.reload(mode_viewer);
mview = mode_viewer.ModeViewer(isheet, modes, lambdas, amplitude=5)
# mview.showScalarField(rod_colors)
mview.show()

In [ ]:
base_energy = isheet.energy(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
g = isheet.gradient(energyType = inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
H = isheet.hessian(energyType = inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
def H_quadratic(x):
    return x @ H.apply(x) / (x @ x) * 0.5

In [ ]:
save_vars = isheet.getVars()

In [ ]:
isheet.setVars(save_vars)

In [ ]:
isheet.energy(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
energies = []

In [ ]:
def taylor_expansion_test(v):
    base_energy = isheet.energy(inflation.InflatableSheet.EnergyType.Elastic)
    g = isheet.gradient(energyType = inflation.InflatableSheet.EnergyType.Elastic)
    H = isheet.hessian(energyType = inflation.InflatableSheet.EnergyType.Elastic)
    def H_quadratic(x):
        return x @ H.apply(x) * 0.5
    save_vars = isheet.getVars()
    
    energies = []
    taylor_expansion_energies = []
    error = []
    steps = np.logspace(-9, 1, 100)
    hessian_quadratic_form = []
    for epsilon in steps:
        isheet.setVars(save_vars + epsilon * v)
        ground_truth_energy = isheet.energy(inflation.InflatableSheet.EnergyType.Elastic)
        taylor_expansion_energy = base_energy + g @ (epsilon * v) + H_quadratic(epsilon * v)
        hessian_quadratic_form.append(H_quadratic(epsilon * v))
        error.append((taylor_expansion_energy - ground_truth_energy) / ground_truth_energy)
        energies.append(ground_truth_energy)
        taylor_expansion_energies.append(taylor_expansion_energy)

    isheet.setVars(save_vars)


    plt.figure(figsize=(15, 4))
    plt.subplot(1, 4, 1)
    plt.title('Elastic Energy formula')
    plt.xlabel('Epsilon')
    plt.ylabel('Energy')
    plt.loglog(steps, energies)
    plt.grid()
    plt.tight_layout()
    
    plt.subplot(1, 4, 2)
    plt.title('Elastic Energy taylor expansion')
    plt.xlabel('Epsilon')
    plt.ylabel('Energy')
    plt.loglog(steps, taylor_expansion_energies)
    plt.tight_layout()
    plt.subplot(1, 4, 3)
    plt.title('Elastic Energy error')
    plt.xlabel('Epsilon')
    plt.ylabel('Error')
    plt.loglog(steps, error)
    plt.tight_layout()
    
    plt.subplot(1, 4, 4)
    plt.title('Elastic Energy Hessian quadratic form')
    plt.xlabel('Epsilon')
    plt.ylabel('Hessian quadratic form')
    plt.loglog(steps, hessian_quadratic_form)
    plt.grid()
    plt.tight_layout()

In [ ]:
taylor_expansion_test(ortho_x)

In [ ]:
taylor_expansion_test(ortho_y)

In [ ]:
taylor_expansion_test(modes[:, 10])

In [ ]:
steps[10]

In [ ]:
H_quadratic(ortho_x * steps[10])

In [ ]:
g @ (ortho_x * steps[10])

In [ ]:
def get_rotation_matrix(theta):
    return np.array([[np.cos(theta), -np.sin(theta), 0],
                     [np.sin(theta), np.cos(theta), 0],
                     [0, 0, 1]])

In [ ]:
def rotation_matrix(axis, theta):
    """
    Return the rotation matrix associated with counterclockwise rotation about
    the given axis by theta radians.
    """
    axis = np.asarray(axis)
    axis = axis / np.sqrt(np.dot(axis, axis))
    a = np.cos(theta / 2.0)
    b, c, d = -axis * np.sin(theta / 2.0)
    aa, bb, cc, dd = a * a, b * b, c * c, d * d
    bc, ad, ac, ab, bd, cd = b * c, a * d, a * c, a * b, b * d, c * d
    return np.array([[aa + bb - cc - dd, 2 * (bc + ad), 2 * (bd - ac)],
                     [2 * (bc - ad), aa + cc - bb - dd, 2 * (cd + ab)],
                     [2 * (bd + ac), 2 * (cd - ab), aa + dd - bb - cc]])

In [ ]:
vertices = np.reshape(isheet.getVars(), (int(isheet.numVars() / 3), 3))

In [ ]:
new_vertices = (rotation_matrix((1, -1, 0), 0.8 * np.pi) @ vertices.transpose()).transpose()

In [ ]:
isheet.setVars(new_vertices.flatten())

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
isheet.energy(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
isheet.energy(inflation.InflatableSheet.EnergyType.Elastic)

In [ ]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])

In [ ]:
viewer.update()

In [ ]:
import fd_validation

In [ ]:
np.random.random(isheet.numVars())

In [ ]:
isheet.setVars(isheet.getVars() + 0.001 * np.random.random(isheet.numVars()))

In [ ]:
fd_validation.gradConvergencePlot(isheet, customArgs={"energyType":inflation.InflatableSheet.EnergyType.Elastic}, epsilons=np.logspace(-10, -3, 100))

In [ ]:
fd_validation.hessConvergencePlot(isheet, customArgs={"energyType":inflation.InflatableSheet.EnergyType.Elastic}, epsilons=np.logspace(-10, -3, 100))

### Repeat the inflation, this time recording it to a video
Requires `MeshFEM`'s `OffscreenRenderer` to be successfully built.

In [ ]:
import time, vis
benchmark.reset()
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 1
opts.niter = 200
framerate = 5 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, [3, 4, 5], opts, callback=cb)
benchmark.report()

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, np.array(fuseMarkers) != 0)
ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)
ipu.sheet.pressure = 1
from tri_mesh_viewer import OffscreenTriMeshViewer
oview = OffscreenTriMeshViewer(ipu, width=2000, height=2000, wireframe=True)

benchmark.reset()
opts.niter=100
oview.recordStart('cc_inflate.mp4')

cr = inflation.inflation_newton(ipu, [3, 4, 5], opts,
                                callback=lambda it: oview.update())
benchmark.report()
oview.recordStop()

In [ ]:
# Some basic statistics for the deformation
from matplotlib import pyplot as plt
strains = utils.getStrains(isheet)[:, 0]
plt.hist(strains, 60);
plt.xlabel('Principal stretch $\\lambda_0$')
print(np.median(strains))

### Analyze curvature of the inflated surface

In [ ]:
isa = inflation.InflatedSurfaceAnalysis(isheet)
curvature = isa.curvature()
metric = isa.metric()

In [ ]:
import matplotlib, vis
from tri_mesh_viewer import TriMeshViewer
isurf = isa.inflatedSurface()
metric_vf = vis.fields.VectorField(isurf, metric.sigma_2[:, None] * metric.left_stretch, vmin=0, vmax=1.0,
                                   align=vis.fields.VectorAlignment.CENTER, colormap=matplotlib.cm.viridis,
                                   glyph=vis.fields.VectorGlyph.CYLINDER)

viewer2 = TriMeshViewer(isurf, width=768, height=640, scalarField=vis.fields.ScalarField(isurf, curvature.meanCurvature(), colormap=matplotlib.cm.coolwarm), vectorField=metric_vf)
viewer2.showWireframe()
viewer2.show()